# Aula 1 (2026.2) - POO em JavaScript e Wireshark: a ponte até o Modbus TCP

## Objetivo da aula

Construir, em duplas, um cliente e um servidor Modbus TCP em JavaScript usando classes, rodar essa comunicação entre duas máquinas da bancada, e capturar e interpretar o tráfego no Wireshark. A turma já conhece Modbus RTU (mestre, escravo, registradores, CRC) mas ainda não viu TCP/IP nem JavaScript. Esta aula não substitui a aula de redes que vocês ainda vão ter sobre TCP/UDP; ela mostra, de forma aplicada ao industrial, o que está por trás da porta 502 antes de vocês verem a teoria completa de camada 4 em sala. Como ninguém da turma programou em JavaScript antes, o material explica cada peça de sintaxe nova antes de usá-la — não é só POO, é POO **e** a linguagem, juntas.

## Formato da aula

- Bancada: switch gerenciado disponível. Sem CLP físico hoje — o servidor Modbus TCP também vai ser escrito em JavaScript, rodando em outro notebook.
- Cada dupla tem um computador. Na parte de rede, duas duplas se juntam formando um **quarteto**: uma dupla assume o papel de **servidor**, a outra de **cliente**.

## Pré-requisitos técnicos

Os computadores do laboratório hoje não têm nada instalado. Por isso a aula abre com um bloco de configuração de ambiente (Node.js, Wireshark, os arquivos do projeto), feito em paralelo em cada dupla no próprio computador. Confirmado: os alunos têm permissão de administrador nas máquinas, e a internet do laboratório é cabeada e boa.

## Roteiro em blocos

- **Bloco 0 — Configuração do ambiente**: instalar Node.js, Wireshark, baixar o projeto e rodar `npm install`.
- **Bloco 1 — POO em JavaScript**: sintaxe básica da linguagem (variáveis, tipos, funções, template literals, arrays, objetos), depois classe, objeto, atributo, método, encapsulamento, abstração, herança e polimorfismo.
- **Bloco 2 — Construindo o cliente e o servidor Modbus TCP**: as peças de JavaScript assíncrono (módulos, callbacks, Promise, async/await), depois duas classes completas (`ClienteModbusTCP` e `ServidorModbusTCP`), com leitura e escrita de registradores, testadas primeiro em localhost.
- **Bloco 3 — Rede: IP, porta e handshake TCP**: só o necessário para entender o que vai aparecer no Wireshark.
- **Bloco 4 — Bancada: montagem e captura**: ligar os notebooks, rodar servidor e cliente entre máquinas diferentes, capturar e interpretar no Wireshark, e ver de fora com espelhamento de porta.

Guarde este ponto para o fechamento: tudo que vocês vão ver no Wireshark hoje está em texto claro. O Modbus não pergunta quem está do outro lado da conexão.

---
# Bloco 0 — Configuração do ambiente

Os computadores do laboratório hoje não têm nada instalado. Este bloco roda em paralelo em cada dupla, no próprio computador, antes de qualquer conceito de POO.

Ordem recomendada: primeiro dispare os dois downloads maiores (Node.js e Wireshark), porque a instalação em si é rápida — o que demora é buscar o instalador na internet.

## 1. Instalar o Node.js

1. Abra o navegador e acesse `https://nodejs.org`.
2. Clique no botão que mostra a versão **LTS** (Long Term Support, a versão estável recomendada).
3. Quando o download terminar, abra o arquivo `.msi` baixado.
4. Na instalação, clique **Next → Next → Install → Finish**, deixando todas as opções padrão marcadas.

**Verificando se instalou:** abra o terminal (`Win + R`, digite `cmd`, Enter) e digite:

```
node --version
npm --version
```

Se aparecer um número de versão em cada linha (por exemplo `v20.11.0` e `10.2.4`), está instalado. Se aparecer erro de comando não reconhecido, feche e abra o terminal de novo — às vezes o `PATH` só atualiza depois de reiniciar o terminal.

## 2. Instalar o Wireshark

1. Acesse `https://www.wireshark.org/download.html`.
2. Baixe o instalador do Windows (o site detecta o sistema automaticamente).
3. Abra o instalador. Clique **Next** nas primeiras telas, aceite a licença.
4. Em algum ponto da instalação vai aparecer uma tela pedindo para instalar o **Npcap**. Deixe marcado e clique **Install** também nessa etapa — o Npcap é o driver que faz a captura de pacotes funcionar. Sem ele, o Wireshark abre mas não consegue capturar nada.
5. Siga até **Finish**. Normalmente não pede reinício do computador, mas se pedir, reinicie antes de seguir.

**Verificando se instalou:** abra o Wireshark pelo menu Iniciar. Se a tela inicial mostrar uma lista de interfaces de rede com um gráfico de atividade ao lado de cada uma, o Npcap está funcionando.

## 3. Baixar os arquivos do projeto

1. Acesse `https://github.com/PortaDoCeu/MonitoriasRedes` no navegador.
2. Clique no botão verde **Code**.
3. Clique em **Download ZIP**.
4. Extraia o arquivo baixado (botão direito no `.zip` → **Extrair tudo**) para uma pasta de fácil acesso, por exemplo a Área de Trabalho.

Se alguém da dupla já tiver o Git instalado, também funciona:

```
git clone https://github.com/PortaDoCeu/MonitoriasRedes.git
```

## 4. Instalar as dependências do projeto

1. Abra o terminal dentro da pasta que você acabou de extrair. No Windows, com a pasta aberta no Explorador de Arquivos, digite `cmd` na barra de endereço e pressione Enter — isso abre o terminal já na pasta certa.
2. Rode:

```
npm install
```

3. Isso lê o `package.json` do projeto e baixa a biblioteca `jsmodbus`, que vamos usar no Bloco 2. Pode levar alguns segundos. Esse `npm install` só precisa rodar uma vez; todo o código que vocês vão escrever depois pode ficar na mesma pasta.

## 5. Checklist antes de seguir para o Bloco 1

Cada dupla confirma, nesta ordem:

- [ ] `node --version` mostra um número de versão.
- [ ] `npm --version` mostra um número de versão.
- [ ] O Wireshark abre e lista as interfaces de rede.
- [ ] A pasta do projeto foi extraída e o terminal abre dentro dela.
- [ ] `npm install` terminou sem erro em vermelho.

Se alguma dupla travar em algum desses passos, siga em frente com o resto da turma — o Bloco 1 não depende do Wireshark nem do `npm install`, só do Node instalado. Dá pra resolver o Wireshark ou o `npm install` de uma dupla específica em paralelo, enquanto a aula segue.

---
# Bloco 1 — POO em JavaScript

## 1. O que é POO e por que serve aqui

POO é uma forma de organizar um programa em torno de **objetos** que representam coisas do mundo real. Em redes industriais, isso é útil porque conseguimos representar sensores, CLPs, IHMs e protocolos como objetos com atributos (dados) e métodos (ações).

Hoje o objeto que a gente vai construir é um **cliente Modbus TCP** e um **servidor Modbus TCP**: cada um tem atributos (IP, porta, registradores) e métodos (conectar, ler, escrever). Os quatro pilares da POO — encapsulamento, abstração, herança e polimorfismo — vão aparecer um a um antes de chegar lá, cada um com um exemplo rodável.

## 1.5 Antes de escrever a primeira classe: sintaxe básica do JavaScript

A turma nunca programou em JavaScript antes. Antes de qualquer classe, vale conhecer as peças de sintaxe que vão aparecer o tempo todo no restante da aula. Isso não é POO ainda, é só "como se escreve" em JavaScript.

### Declarando variáveis: const e let

```js
const nome = "CLP Linha 1";
let contador = 0;
contador = contador + 1; // ok, "let" pode ser reatribuído
nome = "outro nome";     // erro: "const" não pode ser reatribuído
```

- `const`: a variável não pode receber outro valor depois de criada. É a opção padrão.
- `let`: a variável pode ser reatribuída depois. Use só quando o valor realmente precisa mudar (um contador, por exemplo).

Não existe mais `var` em código moderno — se você vir exemplos antigos de JavaScript na internet usando `var`, ignore, hoje se usa `const` e `let`.

### Tipos, sem declarar tipo

Diferente do TypeScript (usado no período passado) e de linguagens como C, o JavaScript não pede para você escrever o tipo da variável. Os tipos existem, só não são declarados:

| Tipo | Exemplo | Observação |
|---|---|---|
| `string` | `"Modbus TCP"` ou `` `Modbus TCP` `` | texto, entre aspas ou crase |
| `number` | `42`, `3.14`, `502` | não existe distinção entre inteiro e decimal |
| `boolean` | `true`, `false` | verdadeiro ou falso |

### Função

```js
function somar(a, b) {
  return a + b;
}

console.log(somar(2, 3)); // 5
```

`function nomeDaFuncao(parametros) { ... }` declara uma função. `return` devolve um valor para quem chamou. **Dentro de uma classe**, como vocês vão ver já na próxima seção, um método não repete a palavra `function`: só o nome, os parênteses e as chaves.

### console.log

É o "print" do JavaScript: imprime qualquer coisa no terminal. Vocês vão usar isso o tempo todo para ver o que o código está fazendo.

### Comparação: === em vez de ==

```js
5 === 5     // true
5 === "5"   // false, tipos diferentes
5 == "5"    // true, mas evite usar "=="
```

Use sempre `===` (compara valor e tipo) em vez de `==` (só compara valor, e pode dar resultado surpreendente).

### Template literals: strings com variável dentro

Strings entre crase (`` ` ``, não confundir com aspas simples `'`) permitem colocar `${expressao}` dentro, que vira o valor daquela expressão:

```js
const nome = "CLP1";
const ip = "192.168.0.10";
console.log(`Dispositivo: ${nome} (${ip})`);
// Dispositivo: CLP1 (192.168.0.10)
```

Isso substitui concatenar strings com `+` (`"Dispositivo: " + nome + " (" + ip + ")"`, mais difícil de ler). Toda vez que aparecer uma string entre crase com `${...}` dentro, é isso.

### Objetos: agrupar valores com chave e valor

```js
const endereco = { host: "192.168.0.10", porta: 502 };
console.log(endereco.host); // 192.168.0.10
```

Um objeto agrupa vários valores nomeados entre chaves `{ }`, no formato `chave: valor`. Isso vai aparecer quando passarmos configurações para a biblioteca Modbus, por exemplo `{ host: "...", port: 502 }`.

### Arrays e o laço for...of

```js
const numeros = [10, 20, 30];

for (const numero of numeros) {
  console.log(numero);
}
```

Um array é uma lista de valores entre colchetes `[ ]`. `for (const item of lista) { ... }` percorre cada item da lista, um de cada vez. É o jeito mais comum de processar uma lista inteira em JavaScript.

### Operador ternário

```js
const status = conectado ? "Conectado" : "Desconectado";
```

É um jeito compacto de escrever "se/senão" que produz um valor: `condicao ? valorSeVerdadeiro : valorSeFalso`. Equivale a:

```js
let status;
if (conectado) {
  status = "Conectado";
} else {
  status = "Desconectado";
}
```

Com essas peças, dá para ler a primeira classe.

## 2. Classe e objeto em JavaScript

Uma **classe** é o molde. Um **objeto** é uma instância criada a partir desse molde.

Diferente do TypeScript (que a monitoria usou no período passado), o JavaScript não tem tipos declarados nos atributos. A sintaxe de classe é parecida, só sem as anotações de tipo.

Dentro de uma classe, o `constructor` é um método especial que roda automaticamente toda vez que um objeto é criado com `new`. Os outros métodos são escritos como `nomeDoMetodo(parametros) { ... }`, sem a palavra `function` na frente — é a única diferença em relação à função "solta" que apareceu na seção anterior.

**Onde rodar:** abra o terminal (no VS Code, menu **Terminal → New Terminal**), crie um arquivo `dispositivo.js` e rode com:

```bash
node dispositivo.js
```

Todo código desta aula roda assim: salva o arquivo `.js` e executa com `node nome-do-arquivo.js`. Não precisa compilar nada.

In [ ]:
class DispositivoRede {
  constructor(nome, ip, protocolo) {
    this.nome = nome;
    this.ip = ip;
    this.protocolo = protocolo;
  }

  apresentar() {
    return `Dispositivo: ${this.nome} | IP: ${this.ip} | Protocolo: ${this.protocolo}`;
  }
}

const clpLinha1 = new DispositivoRede("CLP Linha 1", "192.168.0.10", "Modbus TCP");
const ihmPrincipal = new DispositivoRede("IHM Principal", "192.168.0.20", "PROFINET");

console.log(clpLinha1.apresentar());
console.log(ihmPrincipal.apresentar());


### Leitura do exemplo

- `nome`, `ip` e `protocolo` são **atributos**: guardados em `this` dentro do `constructor`.
- `apresentar()` é um **método**: uma função que pertence à classe.
- `clpLinha1` e `ihmPrincipal` são **objetos**: instâncias criadas com `new`.

**Exercício da dupla:** crie um arquivo `sensor.js` com uma classe `Sensor` que tenha os atributos `nome` e `unidade`, e um método `lerValor()` que retorna uma string como `"Temperatura do Forno: 36.5 °C"`. Adicione também um método `estaEmAlerta(limite)` que retorna `true` ou `false` comparando um valor fixo com o parâmetro `limite`. Rode com `node sensor.js` e confira a saída no terminal.

## 3. Encapsulamento com campo privado

O JavaScript não tem as palavras `public`/`private`/`protected` como o TypeScript. Em vez disso, desde versões recentes do Node, existe o **campo privado**, escrito com `#` na frente do nome. Um campo com `#` só pode ser lido ou alterado de dentro da própria classe.

Isso é útil quando queremos que o próprio objeto controle seu estado interno, em vez de deixar qualquer código de fora alterar diretamente.

In [ ]:
class CLP {
  #conectado;

  constructor(nome) {
    this.nome = nome;
    this.#conectado = false;
  }

  conectar() {
    this.#conectado = true;
  }

  obterStatus() {
    return this.#conectado ? "Conectado" : "Desconectado";
  }
}

const clp = new CLP("CLP Misturador");
console.log(clp.obterStatus());
clp.conectar();
console.log(clp.obterStatus());

// A linha abaixo dá erro se descomentada: campo privado não pode
// ser acessado de fora da classe.
// console.log(clp.#conectado);


No exemplo do `CLP`, `#conectado` só pode ser lido ou mudado por métodos da própria classe. Quem usa o objeto `clp` só enxerga `obterStatus()` e `conectar()`, não o campo interno. Isso é encapsulamento: o objeto expõe um comportamento (conectar, obter status), não o dado bruto.

## 4. Abstração e herança

**Abstração** é definir um contrato: uma classe base diz *o que* toda classe filha precisa saber fazer, sem dizer *como*. O TypeScript tem a palavra `abstract` para isso; o JavaScript puro não tem essa palavra reservada. O jeito idiomático de expressar a mesma ideia é criar um método na classe base que lança um erro se ninguém o substituir:

```js
conectar() {
  throw new Error("cada protocolo precisa implementar conectar()");
}
```

Isso força qualquer subclasse a implementar `conectar()` do seu próprio jeito — se esquecer, o erro aparece na hora de usar, não deixa passar em silêncio.

**Herança** permite criar uma classe nova a partir de uma classe base, com `extends`, reaproveitando estrutura e criando comportamento específico em cada subclasse.

In [ ]:
class ProtocoloIndustrial {
  constructor(nome) {
    this.nome = nome;
  }

  conectar() {
    throw new Error("cada protocolo precisa implementar conectar()");
  }
}

class ModbusTCP extends ProtocoloIndustrial {
  conectar() {
    return `Conectando via ${this.nome} na porta 502`;
  }
}

class Profinet extends ProtocoloIndustrial {
  conectar() {
    return `Conectando via ${this.nome} em rede Ethernet industrial`;
  }
}

const protocolos = [
  new ModbusTCP("Modbus TCP"),
  new Profinet("PROFINET"),
];

for (const protocolo of protocolos) {
  console.log(protocolo.conectar());
}


### Onde está cada pilar neste exemplo

- **Abstração**: `ProtocoloIndustrial` define que todo protocolo tem um `conectar()`, sem dizer como cada um se conecta de fato.
- **Herança**: `ModbusTCP` e `Profinet` herdam de `ProtocoloIndustrial` com `extends`.
- **Encapsulamento**: cada objeto guarda seu próprio `nome`.

Falta só o quarto pilar.

## 5. Polimorfismo

Polimorfismo é chamar o **mesmo método**, pelo **mesmo nome**, em objetos de classes diferentes, e cada um responder do seu próprio jeito. No exemplo anterior, isso já aconteceu: o laço chamou `protocolo.conectar()` para cada item da lista, sem saber se era `ModbusTCP` ou `Profinet`, e cada um respondeu diferente.

Vale ver isso de novo com um exemplo focado só nesse pilar, com uma classe abstrata `Sensor` e duas subclasses concretas.

In [ ]:
class Sensor {
  constructor(nome, unidade) {
    this.nome = nome;
    this.unidade = unidade;
  }

  lerValor() {
    throw new Error("cada sensor precisa implementar lerValor()");
  }
}

class SensorTemperatura extends Sensor {
  lerValor() {
    return `${this.nome}: 36.5 ${this.unidade}`;
  }
}

class SensorPressao extends Sensor {
  lerValor() {
    return `${this.nome}: 5.2 ${this.unidade}`;
  }
}

const sensores = [
  new SensorTemperatura("Temperatura do Forno", "°C"),
  new SensorPressao("Pressao da Tubulacao", "bar"),
];

for (const sensor of sensores) {
  console.log(sensor.lerValor());
}


O laço `for` chama `sensor.lerValor()` sem nunca perguntar "isso é uma temperatura ou uma pressão?". Cada objeto já sabe responder por conta própria. É isso que permite, por exemplo, colocar um `SensorTemperatura`, um `SensorPressao` e amanhã um `SensorNivel` na mesma lista e tratar todos igual no código que os consome.

**Exercício da dupla:** crie uma classe `DispositivoModbus` com os atributos `ip`, `porta` e `unitId`, e um método `descrever()` que retorna uma string juntando os três. Esse exercício não é decorativo: no Bloco 2, o cliente e o servidor Modbus TCP que vocês vão construir usam exatamente esses três atributos.

---
# Bloco 2 — Construindo o cliente e o servidor Modbus TCP

## 1. O ponto de partida: um script solto

Este repositório já tem um exemplo de cliente Modbus TCP em TypeScript, em `src/index.ts`, usando a biblioteca `jsmodbus`. Ele conecta em um IP fixo, lê 10 holding registers e imprime o resultado. Hoje vamos escrever a versão em JavaScript puro, mas em vez de scripts soltos, vamos organizar cliente e servidor como **classes** — reaproveitando a classe `DispositivoModbus` que a dupla esboçou no fim do Bloco 1.

**Por que virar classe:** um script solto só sabe fazer uma coisa, do jeito que foi escrito. Uma classe `ClienteModbusTCP` pode ser instanciada várias vezes, com IPs diferentes, e reaproveitada em outros contextos.

A biblioteca `jsmodbus` já foi instalada no Bloco 0, junto com o `npm install` do projeto. Não precisa instalar de novo.

## Antes do código: peças novas de sintaxe

Conectar em rede é diferente de somar dois números: a resposta não chega na hora. O código do cliente e do servidor Modbus TCP usa algumas construções de JavaScript que ainda não apareceram, e todas existem para lidar com isso. Vale entender cada uma antes de ler a classe inteira.

### require() e module.exports — importando código de outro arquivo

Node.js organiza código em módulos: cada arquivo `.js` pode **exportar** algo com `module.exports = ...`, e outro arquivo **importa** isso com `require("...")`.

```js
// arquivo utilitarios.js
function dobrar(x) {
  return x * 2;
}
module.exports = dobrar;
```

```js
// outro arquivo, na mesma pasta
const dobrar = require("./utilitarios.js");
console.log(dobrar(5)); // 10
```

`require("net")`, sem `./` na frente, importa um módulo que já vem embutido no Node (não precisa instalar nada). `require("./cliente.js")`, com `./` na frente, importa um arquivo do próprio projeto de vocês.

### Desestruturação — pegar só uma peça de dentro de algo

```js
const { Socket } = require("net");
```

Isso não cria uma variável chamada `{ Socket }`. As chaves dizem: "de dentro do que `require('net')` devolve, pegue só a propriedade chamada `Socket` e guarde numa variável com esse mesmo nome". É equivalente a escrever:

```js
const net = require("net");
const Socket = net.Socket;
```

só que mais direto. A mesma ideia funciona pra qualquer objeto, não só para `require`.

### Arrow function — outro jeito de escrever uma função

```js
const dobrar = (x) => x * 2;
```

é outra forma de escrever

```js
function dobrar(x) {
  return x * 2;
}
```

Arrow functions (`=>`) aparecem muito quando a função é curta e é passada como argumento para outra função — exatamente o caso de um callback, que é o próximo tópico.

### Callback — uma função entregue para ser chamada depois

Um callback é uma função que você entrega para outra parte do código, para ser chamada quando algo acontecer, e não agora. Exemplo:

```js
socket.on("connect", () => {
  console.log("conectou!");
});
```

Isso lê: "quando o evento `connect` acontecer neste `socket`, rode esta função". A função entre `=> { ... }` não roda na hora que essa linha é executada — ela fica guardada, esperando o evento.

### Promise — representar algo que ainda vai acontecer

Abrir uma conexão de rede não é instantâneo. O JavaScript não trava o programa esperando; em vez disso, ele usa uma `Promise`: um objeto que representa "uma resposta que ainda vai chegar, com sucesso ou com erro".

```js
function esperarConexao(socket) {
  return new Promise((resolve, reject) => {
    socket.on("connect", resolve);
    socket.on("error", reject);
  });
}
```

Uma Promise recém-criada recebe uma função com dois parâmetros, `resolve` e `reject` — também dois callbacks, só que fornecidos pelo próprio JavaScript. Chamar `resolve(valor)` marca a Promise como "deu certo"; chamar `reject(erro)` marca como "deu errado". No exemplo acima, é só passar `resolve` e `reject` diretamente como os callbacks dos eventos `"connect"` e `"error"`.

### async / await — esperar uma Promise como se fosse código normal

Encadear Promises com `.then(...)` para cada passo fica difícil de ler quando há vários passos em sequência. `async`/`await` é uma sintaxe que deixa código assíncrono parecido com código sequencial comum:

```js
async function main() {
  await esperarConexao(socket);
  console.log("só chega aqui depois que a conexao terminou");
}
```

Regras práticas:

- Toda função que usa `await` dentro precisa ser declarada com `async` na frente.
- `await umaPromise` pausa a função ali (sem travar o resto do programa) até a Promise terminar, e devolve o valor de `resolve(...)`. Se a Promise terminar com `reject(...)`, o `await` lança um erro que pode ser capturado com `try`/`catch` ou com `.catch(...)` fora da função.
- Toda função `async` devolve uma Promise automaticamente. É por isso que dá para escrever `main().catch((erro) => console.error(erro))`: se algo dentro de `main` der erro, o `.catch` no final recebe esse erro.

Com essas seis peças (`require`/`module.exports`, desestruturação, arrow function, callback, Promise, `async`/`await`), o código do cliente e do servidor fica legível.

## 2. A classe ClienteModbusTCP

Crie um arquivo `cliente.js` com o código abaixo. Ele encapsula o `Socket` do Node e o cliente `jsmodbus` dentro de uma classe, com três métodos: `conectar()`, `lerRegistradores()` e `escreverRegistradores()`.

In [ ]:
const { Socket } = require("net");
const Modbus = require("jsmodbus");

class ClienteModbusTCP {
  constructor(ip, porta) {
    this.ip = ip;
    this.porta = porta;
    this.socket = new Socket();
    this.client = new Modbus.client.TCP(this.socket);
  }

  conectar() {
    return new Promise((resolve, reject) => {
      this.socket.on("connect", resolve);
      this.socket.on("error", reject);
      this.socket.connect({ host: this.ip, port: this.porta });
    });
  }

  async lerRegistradores(enderecoInicial, quantidade) {
    const resposta = await this.client.readHoldingRegisters(enderecoInicial, quantidade);
    return resposta.response.body.valuesAsArray;
  }

  async escreverRegistradores(enderecoInicial, valores) {
    await this.client.writeMultipleRegisters(enderecoInicial, valores);
  }

  encerrar() {
    this.socket.end();
  }
}

module.exports = ClienteModbusTCP;


### Leitura do código

- `constructor(ip, porta)`: guarda IP e porta como atributos, cria o `Socket` e o cliente `jsmodbus` por cima dele. É a mesma ideia da classe `DispositivoModbus` do Bloco 1, só que com comportamento de verdade.
- `conectar()`: devolve uma `Promise` que resolve quando o evento `"connect"` do socket dispara. Repare que `resolve` e `reject` são passados direto como os callbacks de `"connect"` e `"error"` — exatamente o padrão mostrado na seção anterior. É assim que sabemos que o TCP handshake terminou.
- `lerRegistradores(enderecoInicial, quantidade)`: chama `readHoldingRegisters`, que monta e envia o pacote Modbus TCP com function code 3 (Read Holding Registers), igual ao que a turma já viu no Aula2 sobre Modbus.
- `escreverRegistradores(enderecoInicial, valores)`: chama `writeMultipleRegisters`, function code 16 (Write Multiple Registers). `valores` é um array de números.
- `encerrar()`: fecha a conexão TCP.

## 3. A classe ServidorModbusTCP

Do lado do servidor, a ideia é a mesma: encapsular o servidor TCP do Node e o servidor `jsmodbus` numa classe, com o bloco de holding registers guardado como campo privado.

Crie um arquivo `servidor.js`:

In [ ]:
const net = require("net");
const Modbus = require("jsmodbus");

class ServidorModbusTCP {
  #holding;
  #netServer;

  constructor(porta, quantidadeRegistradores = 100) {
    this.porta = porta;
    // Cada holding register tem 16 bits (2 bytes), por isso o buffer
    // precisa do dobro de bytes da quantidade de registradores.
    this.#holding = Buffer.alloc(quantidadeRegistradores * 2);
    this.#netServer = net.createServer();

    // Evita que o processo trave com um erro não tratado se o cliente
    // derrubar a conexão de forma abrupta.
    this.#netServer.on("connection", (socket) => {
      socket.on("error", (erro) => {
        console.log("Conexao encerrada pelo cliente:", erro.code);
      });
    });

    new Modbus.server.TCP(this.#netServer, { holding: this.#holding });
  }

  definirRegistrador(indice, valor) {
    this.#holding.writeUInt16BE(valor, indice * 2);
  }

  iniciar() {
    this.#netServer.listen(this.porta, "0.0.0.0", () => {
      console.log(`Servidor Modbus TCP ouvindo na porta ${this.porta}`);
    });
  }
}

module.exports = ServidorModbusTCP;


### O que é um Buffer

`Buffer.alloc(n)` cria um bloco de `n` bytes, todos zerados, em memória. É a forma que o Node usa para representar dados binários brutos — diferente de um array comum, que guarda qualquer tipo de valor, um `Buffer` guarda só bytes (números de 0 a 255). Os registradores Modbus são valores de 16 bits (2 bytes), por isso o construtor aloca `quantidadeRegistradores * 2` bytes: um registrador ocupa duas posições do buffer.

`.writeUInt16BE(valor, deslocamento)` escreve um número de 16 bits dentro do buffer, começando no byte `deslocamento`, no formato **big-endian** (`BE`): o byte mais significativo primeiro. É o mesmo formato que a especificação Modbus usa para transmitir valores de mais de um byte, então não é coincidência — é o motivo de existir esse método com esse nome.

### Leitura do código

- `#holding` e `#netServer` são campos privados: nada fora da classe mexe direto no buffer de registradores ou no servidor TCP bruto. É o mesmo encapsulamento da classe `CLP` do Bloco 1.
- `this.#netServer.on("connection", (socket) => { ... })`: outro callback, como os vistos na seção anterior — roda toda vez que um cliente conecta, não na hora que essa linha é executada.
- `definirRegistrador(indice, valor)`: escreve um valor de 16 bits num registrador específico, calculando o deslocamento em bytes (`indice * 2`, porque cada registrador ocupa 2 bytes).
- `iniciar()`: só agora o servidor de fato começa a escutar a porta.

Note que a classe `ServidorModbusTCP`, por dentro, não sabe nada sobre function codes, MBAP ou os campos do protocolo Modbus TCP — quem cuida disso é a biblioteca `jsmodbus`. A classe só organiza a configuração e expõe uma API simples (`definirRegistrador`, `iniciar`). Isso também é abstração: esconder complexidade que não precisa aparecer para quem usa a classe.

Agora crie um segundo arquivo, `iniciar-servidor.js`, para efetivamente ligar o servidor com alguns valores de teste:

In [ ]:
const ServidorModbusTCP = require("./servidor.js");

const PORTA = 502;
const servidor = new ServidorModbusTCP(PORTA);

servidor.definirRegistrador(0, 100);
servidor.definirRegistrador(1, 200);
servidor.definirRegistrador(2, 300);
servidor.definirRegistrador(3, 400);

servidor.iniciar();


**Sobre a porta 502:** algumas portas abaixo de 1024, como a 502, podem exigir privilégio de administrador dependendo do sistema operacional. Se `node iniciar-servidor.js` der erro de permissão, duas opções:

1. Rodar o terminal como administrador.
2. Trocar `const PORTA = 502;` por `const PORTA = 5020;` (ou outra porta acima de 1024) e usar a mesma porta depois no `testar-cliente.js`.

Como os alunos têm permissão de administrador nas máquinas do laboratório hoje, a primeira opção deve funcionar direto.

## 4. Testando localmente, antes de ir para a rede

Antes de rodar cliente e servidor em máquinas diferentes (Bloco 4), vale testar os dois na mesma máquina, contra `127.0.0.1` (o próprio computador). Isso separa dois tipos de erro possíveis: erro no código versus erro de rede.

Crie um arquivo `testar-cliente.js`:

In [ ]:
const ClienteModbusTCP = require("./cliente.js");

async function main() {
  // Para o teste local, deixe 127.0.0.1. No Bloco 4, isso vira o IP
  // real do notebook que estiver rodando o servidor.
  const cliente = new ClienteModbusTCP("127.0.0.1", 502);

  await cliente.conectar();
  console.log("Conectado ao servidor Modbus TCP");

  const antes = await cliente.lerRegistradores(0, 4);
  console.log("Registradores antes de escrever:", antes);

  await cliente.escreverRegistradores(0, [11, 22, 33, 44]);
  console.log("Escrita concluida");

  const depois = await cliente.lerRegistradores(0, 4);
  console.log("Registradores depois de escrever:", depois);

  cliente.encerrar();
}

main().catch((erro) => console.error("Erro:", erro));


**Rodando o teste local:** em um terminal, rode `node iniciar-servidor.js` e deixe rodando. Em outro terminal (mesma pasta), rode `node testar-cliente.js`. A saída esperada é:

```
Conectado ao servidor Modbus TCP
Registradores antes de escrever: [ 100, 200, 300, 400 ]
Escrita concluida
Registradores depois de escrever: [ 11, 22, 33, 44 ]
```

Os valores mudaram entre o "antes" e o "depois" porque o `escreverRegistradores` (function code 16) realmente alterou o buffer `#holding` dentro do servidor, e a segunda leitura (function code 3) pegou o valor novo. Cliente e servidor, mesmo sendo dois processos separados, se comunicaram só através da rede — nesse teste local, através da interface de loopback do próprio computador.

Se esse teste funcionar, o código está correto e pronto para o Bloco 4. Se não funcionar, é mais fácil resolver agora, contra `127.0.0.1`, do que depois de já estar conectado em outro notebook.

---
# Bloco 3 — Rede: IP, porta e handshake TCP

## 1. Por que isso importa agora

O Modbus RTU que vocês já conhecem identifica o destino pelo **endereço do escravo** no barramento serial. O Modbus TCP identifica o destino por **IP + porta**. Antes de rodar cliente e servidor em máquinas diferentes, cada dupla precisa saber o IP da própria máquina e testar se enxerga a máquina da outra dupla do quarteto.

## 2. Por que TCP, e não UDP

O Modbus TCP escolhe o nome pelo transporte que usa: TCP. Vale entender por que essa escolha faz sentido aqui, mesmo sem entrar na aula completa de camada 4 que a turma ainda vai ter.

| | TCP | UDP |
|---|---|---|
| Conexão | orientado a conexão (handshake antes de trocar dados) | sem conexão, cada pacote é independente |
| Entrega | garante entrega e ordem, retransmite se perder pacote | não garante nada disso |
| Overhead | maior (handshake, confirmações) | menor |
| Uso típico | onde o dado não pode simplesmente sumir | onde velocidade importa mais que garantia |

Um comando Modbus de escrita (function code 16, o que a classe `ClienteModbusTCP` usa em `escreverRegistradores`) muda o estado de um equipamento remoto. Se esse pacote se perdesse no meio do caminho sem ninguém perceber, o operador acharia que o comando foi aplicado quando não foi. O TCP existe exatamente para eliminar essa dúvida: ou o dado chega íntegro e na ordem certa, ou o remetente fica sabendo que precisa reenviar.

## 3. Descobrindo o IP da máquina

**No Windows:**
1. Abra o terminal (`Win + R`, digite `cmd`, Enter).
2. Digite:
   ```
   ipconfig
   ```
3. Procure a linha **"Adaptador Ethernet"** referente à porta conectada no switch e anote o **"Endereço IPv4"**.

Se o notebook já pegou um IP automaticamente (DHCP) e está na mesma faixa de rede da outra dupla, pode seguir direto para o teste de ping. Se não tiver IP nenhum na interface Ethernet, será preciso configurar um IP manual: **Painel de Controle → Rede e Internet → Central de Rede e Compartilhamento → Alterar configurações do adaptador**, clicar com o botão direito na interface Ethernet, **Propriedades → Protocolo IP Versão 4 → Propriedades**, e definir IP e máscara manualmente. Combine com o monitor qual faixa de IP usar nesse caso, porque depende de como a rede da bancada está configurada hoje.

## 4. Testando a conectividade

Com os dois notebooks do quarteto ligados no switch, no terminal de um deles:

```
ping <IP do outro notebook>
```

Se as respostas chegarem (`Resposta de ...`), a rede está funcionando. Se der **tempo limite esgotado**, revise: cabo conectado na porta certa do switch, os dois IPs na mesma faixa de rede, firewall do Windows não bloqueando ICMP.

## 5. O que vai acontecer por baixo: handshake TCP

Antes de qualquer dado Modbus trafegar, o TCP precisa abrir a conexão. Isso acontece em três passos, chamados de **three-way handshake**:

| Passo | Quem envia | O que significa |
|---|---|---|
| 1. SYN | Cliente → Servidor | "quero abrir uma conexão" |
| 2. SYN-ACK | Servidor → Cliente | "recebi, também quero, aqui está minha confirmação" |
| 3. ACK | Cliente → Servidor | "confirmado, conexão aberta" |

Isso é exatamente o que dispara quando o método `conectar()` da classe `ClienteModbusTCP` chama `socket.connect()`. Só depois desse handshake terminar (evento `"connect"`) é que os pacotes Modbus TCP de fato saem.

---
# Bloco 4 — Bancada: montagem e captura

## 1. Organizando os quartetos

Junte-se com outra dupla. Decidam quem vai ser a dupla **servidor** e quem vai ser a dupla **cliente** primeiro (os papéis trocam depois, no passo 6).

## 2. Passo a passo: dupla servidor

1. Confirme o IP da sua máquina com `ipconfig` (Bloco 3).
2. Abra o **Wireshark**.
3. Na tela inicial, escolha a interface de rede Ethernet correta (a mesma que está conectada ao switch) e clique duas vezes nela, ou selecione e clique no botão de tubarão azul no canto superior esquerdo, para **iniciar a captura**.
4. Deixe o Wireshark capturando e, no terminal, rode:
   ```
   node iniciar-servidor.js
   ```
5. Deve aparecer: `Servidor Modbus TCP ouvindo na porta 502` (ou a porta que vocês combinaram no Bloco 2).
6. Passe o **seu IP** para a dupla cliente.

## 3. Passo a passo: dupla cliente

1. Abra `testar-cliente.js` e troque `"127.0.0.1"` pelo IP que a dupla servidor passou.
2. Rode:
   ```
   node testar-cliente.js
   ```
3. Deve aparecer no terminal a mesma saída do teste local do Bloco 2, agora atravessando a rede de verdade:
   ```
   Conectado ao servidor Modbus TCP
   Registradores antes de escrever: [ 100, 200, 300, 400 ]
   Escrita concluida
   Registradores depois de escrever: [ 11, 22, 33, 44 ]
   ```

Se aparecer erro de conexão recusada ou tempo esgotado, confira: o servidor está mesmo rodando, o IP está certo, e não há firewall bloqueando a porta no notebook servidor (no Windows, o primeiro `node iniciar-servidor.js` costuma abrir um pop-up perguntando se permite acesso à rede — clique em **Permitir acesso**).

## 4. Analisando a captura no Wireshark (dupla servidor)

1. Volte para o Wireshark e clique no quadrado vermelho para **parar a captura**.
2. No campo de filtro (barra branca no topo), digite:
   ```
   tcp.port == 502
   ```
   (troque `502` pela porta que vocês usaram, se tiverem mudado)
3. Pressione Enter.

Você deve ver uma sequência parecida com esta:

| # | Origem | Destino | Protocolo | Info |
|---|---|---|---|---|
| 1 | IP do cliente | IP do servidor | TCP | `... [SYN] Seq=0 ...` |
| 2 | IP do servidor | IP do cliente | TCP | `... [SYN, ACK] Seq=0 Ack=1 ...` |
| 3 | IP do cliente | IP do servidor | TCP | `... [ACK] Seq=1 Ack=1 ...` |
| 4 | IP do cliente | IP do servidor | Modbus/TCP | `Query: Trans: ...; Unit: 1, Func: 3: Read Holding Registers` |
| 5 | IP do servidor | IP do cliente | Modbus/TCP | `Response: Trans: ...; Unit: 1, Func: 3: Read Holding Registers` |
| 6 | IP do cliente | IP do servidor | Modbus/TCP | `Query: Trans: ...; Unit: 1, Func: 16: Write Multiple Registers` |
| 7 | IP do servidor | IP do cliente | Modbus/TCP | `Response: Trans: ...; Unit: 1, Func: 16: Write Multiple Registers` |
| 8 | IP do cliente | IP do servidor | Modbus/TCP | `Query: Trans: ...; Unit: 1, Func: 3: Read Holding Registers` |
| 9 | IP do servidor | IP do cliente | Modbus/TCP | `Response: Trans: ...; Unit: 1, Func: 3: Read Holding Registers` |

Os três primeiros pacotes são o handshake TCP do Bloco 3. Depois vêm três operações Modbus TCP seguidas: a primeira leitura, a escrita e a segunda leitura (nessa ordem, porque é essa a ordem no `testar-cliente.js`).

**Clique no pacote 4** (a primeira `Query`) e expanda, no painel do meio, a camada **Modbus/TCP**. Localize:

- **Transaction Identifier**: o número que casa a resposta com o pedido.
- **Unit Identifier**: normalmente `1`.
- **Function Code**: deve aparecer `3, Read Holding Registers`.
- **Reference Number** (ou **Starting Address**): o endereço inicial, `0`.
- **Word Count** (ou **Quantity**): quantos registradores, `4`.

Clique no pacote 5 (`Response`) e confira que os valores `100, 200, 300, 400` aparecem em texto claro.

**Agora clique no pacote 6**, a `Query` com `Func: 16`. Compare com o pacote 4: o Function Code mudou de `3` para `16`, e agora existem campos adicionais — **Byte Count** e os **valores a escrever** (`11, 22, 33, 44`) dentro do próprio pedido, porque escrever exige mandar os dados junto, ao contrário de ler.

Por fim, o pacote 9 (a segunda `Response` de leitura) mostra os valores já atualizados: `11, 22, 33, 44`. O que apareceu na tela do terminal também apareceu, em claro, no fio.

**Ponto para discutir com a dupla:** nada disso está cifrado, nem autenticado. Qualquer pessoa capturando esse tráfego lê o pedido, a resposta, o function code e os valores exatos — inclusive o comando de escrita.

## 5. Vendo de fora: espelhamento de porta (demonstração do monitor)

Até aqui, o Wireshark rodou na própria máquina servidor, uma das partes da comunicação. Isso é o suficiente para o exercício de cada dupla, mas não é como um sistema de detecção de intrusão observa a rede na prática: um IDS normalmente não é nem o cliente nem o servidor, ele fica de fora, olhando o tráfego passar.

O monitor vai demonstrar isso ligando seu próprio notebook, com Wireshark, numa porta do switch configurada para espelhar as portas de um dos quartetos — reaproveitando os comandos de espelhamento já usados na Aula1 do período passado (`create mirror`, `configure mirror ... add port ... ingress-and-egress`, `enable mirror`). Como o switch pode ter sido reconfigurado desde então, confirme o estado atual (`show mirror`) antes de aplicar os comandos.

Guardem este ponto: o espelhamento de porta é exatamente o "grampo" que um sistema de detecção de intrusão usa para enxergar o tráfego de terceiros, sem estar em nenhuma ponta da comunicação. Isso volta a aparecer quando o período chegar na parte de detecção.

## 6. Trocando os papéis

Repitam o processo com os papéis invertidos: quem era cliente vira servidor, e vice-versa. Isso reforça que `cliente.js` e `servidor.js` são só código — qualquer máquina pode rodar qualquer um dos dois papéis.

---
# Fechamento

## Debrief

- O que apareceu em texto claro no Wireshark? IP de origem e destino, function code, endereço, quantidade e os valores lidos **e escritos**.
- O protocolo Modbus TCP verificou, em algum momento, **quem** estava do outro lado da conexão? Não. Qualquer dispositivo que alcance a porta 502 do servidor pode pedir os mesmos registradores, ou escrever neles.
- Se hoje vocês conseguiram escrever num registrador só sabendo o IP e a porta do servidor, o que mais alguém de fora da dupla, na mesma rede, conseguiria fazer com essa mesma informação?

**Semente para a próxima etapa do período:** se o Modbus não autentica quem está perguntando, o que impede alguém de repetir exatamente o mesmo pacote de escrita mais tarde, ou de forjar um pacote parecido com um endereço e valor diferentes? Essa pergunta vai ser respondida quando a turma chegar na parte de ataque controlado (injeção e replay), sempre em bancada isolada.

## Entregável da dupla

1. Print da tela do Wireshark mostrando o pacote de `Query` com `Func: 16` (escrita) com os valores escritos identificados.
2. Os arquivos `cliente.js` e `servidor.js` (ou só um dos dois, dependendo do papel que a dupla assumiu por último) usados na aula.